# 05 – Ensemble, TTA & Transductive Re-ranking (Phase 6)

**Phase 6 exit gate:** Final pipeline beats the best single model on validation. Each added component has a measured positive contribution (ablation table).

**Inputs:** Cached features from notebooks 02 (zero-shot) and 04 (fine-tuned), validation OOF predictions.

In [ ]:
import sys, pathlib
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
sys.path.append("/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

from src.config import DATA_DIR, OUT_DIR, seed_everything
seed_everything(42)

from src.data import load_csvs, parse_val_ground_truth
from src.features import load_features, load_text_features
from src.text_matching import build_surface_form_dict, TextMatcher
from src.metrics import evaluate, print_metrics
from src.ensemble import LogitEnsemble, temperature_scale, fit_temperature, blend_zeroshot_finetuned
from src.rerank import rerank, rank_for_category

dfs = load_csvs(DATA_DIR)
meta        = dfs["category_metadata"]
val_gallery = dfs["val_gallery"]
val_queries = dfs["val_queries"]
test_gallery= dfs["test_gallery"]
test_queries= dfs["test_queries"]
val_gt      = parse_val_ground_truth(dfs["val_gt"])

surface_forms = build_surface_form_dict(meta)
matcher       = TextMatcher(surface_forms, meta_df=meta)
cat_id_list   = meta["category_id"].tolist()
cat_id2idx    = {int(c): i for i, c in enumerate(cat_id_list)}

# Load cached features for all models to ensemble
MODEL_TAGS = ["vitl14_openai", "siglip_so400m"]  # add fine-tuned tag when available
val_features: dict = {}
val_scores: dict   = {}
test_features: dict = {}
test_scores: dict   = {}

for tag in MODEL_TAGS:
    feat_dir = pathlib.Path(OUT_DIR) / "features" / tag
    X_v, val_ids = load_features(feat_dir, "val", gallery_df=val_gallery)
    T_v, _       = load_text_features(feat_dir, tag="text")
    val_features[tag]  = X_v
    val_scores[tag]    = X_v @ T_v.T  # (N_val, K) cosine

    X_t, test_ids = load_features(feat_dir, "test", gallery_df=test_gallery)
    T_t, _        = load_text_features(feat_dir, tag="text")
    test_features[tag]  = X_t
    test_scores[tag]    = X_t @ T_t.T

print(f"Loaded features for: {MODEL_TAGS}")

## 1. Per-model temperature calibration

In [ ]:
# Temperature should be fit on train OOF logits, not val labels.
# Using val here as a placeholder – replace with OOF logits from Phase 4.
temperatures = {}
for tag in MODEL_TAGS:
    # TODO: replace with train OOF logits + train labels
    # T_opt = fit_temperature(oof_logits[tag], oof_labels)
    T_opt = 1.0   # placeholder
    temperatures[tag] = T_opt
    print(f"  {tag}: T={T_opt}")

## 2. Log-probability ensemble

In [ ]:
ens = LogitEnsemble()
for tag in MODEL_TAGS:
    ens.add(val_scores[tag], weight=1.0, temperature=temperatures[tag])
S_ens_val = ens.combine()  # (N_val, K)

def score_to_preds(S, query_df, matcher, cat_id2idx, gallery_ids, top=20):
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order = np.argsort(-S[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [gallery_ids[j] for j in order[:top]]
    return pred_dict

ens_preds   = score_to_preds(S_ens_val, val_queries, matcher, cat_id2idx, val_ids)
ens_metrics = evaluate(ens_preds, val_gt)
print("Ensemble (uniform weights):")
print_metrics(ens_metrics)

## 3. Balanced re-ranking (Sinkhorn)

In [ ]:
# Ablation: validate on validation gallery (capacity=10) before applying to test
ablation_results = []

for use_sink, use_knn in [(False, False), (True, False), (False, True), (True, True)]:
    label = f"sinkhorn={use_sink}, knn={use_knn}"
    # Average val features across models for kNN smoothing
    avg_val_feats = np.mean([val_features[t] for t in MODEL_TAGS], axis=0)
    avg_val_feats /= np.linalg.norm(avg_val_feats, axis=1, keepdims=True) + 1e-9

    S_rr = rerank(
        S_ens_val,
        image_features=avg_val_feats,
        use_sinkhorn=use_sink,
        use_knn=use_knn,
        sinkhorn_strength=0.5,
        knn_k=5,
        knn_alpha=0.3,
    )
    preds   = score_to_preds(S_rr, val_queries, matcher, cat_id2idx, val_ids)
    metrics = evaluate(preds, val_gt)
    ablation_results.append({"config": label, **metrics})
    print(f"  {label}: R-prec={metrics['r_precision']:.4f}")

print("\nAblation table:")
print(pd.DataFrame(ablation_results)[["config", "r_precision", "map@20", "p@5", "hit@1", "mrr@20"]].to_string(index=False))

## 4. Tune Sinkhorn strength on validation

In [ ]:
from src.rerank import sinkhorn_balance

best_str, best_rp = 0.0, -1.0
for strength in np.arange(0.0, 1.05, 0.1):
    S_rr = sinkhorn_balance(S_ens_val, n_iter=20, strength=float(strength))
    preds = score_to_preds(S_rr, val_queries, matcher, cat_id2idx, val_ids)
    rp = evaluate(preds, val_gt)["r_precision"]
    print(f"  strength={strength:.1f}  R-prec={rp:.4f}")
    if rp > best_rp:
        best_rp, best_str = rp, float(strength)

print(f"\nBest Sinkhorn strength: {best_str:.1f}  (R-prec={best_rp:.4f})")